# Planners-2-PDDL-Basics

**Navigation** : [Index](../../README.md) | [<< Introduction](Planners-1-Introduction.ipynb) | [State Space >>](Planners-3-State-Space.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Lire et comprendre la syntaxe PDDL (Planning Domain Definition Language)
2. Ecrire un fichier domaine PDDL avec predicats et actions
3. Ecrire un fichier probleme PDDL avec objets, etat initial et but
4. Utiliser `unified-planning` pour manipuler des modeles PDDL en Python
5. Comprendre les schemas d'action et leur instanciation

### Prerequis
- Python 3.9+
- Connaissances basiques en logique propositionnelle
- Avoir suivi le notebook [Planners-0-Setup](../00-Environment/Planners-0-Setup.ipynb)

### Duree estimee : 40 minutes

---

## 1. Introduction a PDDL

**PDDL** (Planning Domain Definition Language) est le langage standard pour decrire des problemes de planification automatique. Developpe en 1998 pour la premiere competition internationale de planification (IPC), il est base sur le modele STRIPS avec des extensions.

### Pourquoi PDDL ?

| Avantage | Description |
|----------|------------|
| **Standardisation** | Format commun pour tous les planificateurs |
| **Separation** | Domaine (reutilisable) vs Probleme (specifique) |
| **Expressivite** | Typage, predicats, actions parametrees |
| **Extensibilite** | Nombreuses extensions (temporel, numerique, etc.) |

### Architecture PDDL

Un probleme de planification en PDDL se compose de **deux fichiers** :

1. **Domaine** (`domain.pddl`) : Definit les types, predicats et actions disponibles
2. **Probleme** (`problem.pddl`) : Definit les objets, l'etat initial et le but

### 1.1 STRIPS : La base de PDDL

PDDL est base sur le modele **STRIPS** (Stanford Research Institute Problem Solver, 1971). Dans STRIPS, une action est definie par :

| Composante | Description |
|------------|-------------|
| **Nom** | Identifiant de l'action |
| **Parametres** | Variables instanciees lors de l'execution |
| **Preconditions** | Conditions necessaires pour executer l'action |
| **Effets** | Changements apres l'execution (ajouts et suppressions) |

#### Exemple STRIPS : Monde des blocs

```
pickup(X)
  P: gripping() ^ clear(X) ^ ontable(X)
  A: gripping(X)
  D: ontable(X) ^ gripping()

putdown(X)
  P: gripping(X)
  A: ontable(X) ^ gripping() ^ clear(X)
  D: gripping(X)

stack(X, Y)
  P: gripping(X) ^ clear(Y)
  A: on(X,Y) ^ gripping() ^ clear(X)
  D: gripping(X) ^ clear(Y)

unstack(X, Y)
  P: gripping() ^ clear(X) ^ on(X,Y)
  A: gripping(X) ^ clear(Y)
  D: on(X,Y) ^ gripping()
```

> **Notation** : P = Preconditions, A = Ajouts (Add), D = Suppressions (Delete)

---

## 2. Structure d'un fichier domaine PDDL

Le fichier **domaine** definit la structure du monde : types d'objets, predicats (proprietes) et actions possibles.

### Syntaxe generale

```pddl
(define (domain nom-du-domaine)
  (:requirements <requirements...>)
  (:types <types...>)
  (:predicates <predicates...>)
  (:action nom-action
    :parameters (<params...>)
    :precondition <precond>
    :effect <effets>)
  ...
)
```

### 2.1 En-tete et requirements

```pddl
(define (domain logistics)
  (:requirements :strips :typing)
  ...
)
```

| Requirement | Description |
|-------------|-------------|
| `:strips` | Support de base STRIPS (requis) |
| `:typing` | Types d'objets hierarchiques |
| `:negative-preconditions` | Preconditions avec `not` |
| `:disjunctive-preconditions` | Preconditions avec `or` |
| `:equality` | Test d'egalite `=` entre objets |
| `:conditional-effects` | Effets conditionnels `when` |
| `:adl` | Combinaison de plusieurs requirements |

### 2.2 Types d'objets

```pddl
(:types 
  location vehicle - object
  truck airplane - vehicle
  city place - location
)
```

**Syntaxe** : `sous-type1 sous-type2 - type-parent`

Les types permettent de contraindre les parametres des actions et de reduire l'espace de recherche.

### 2.3 Predicats

Les **predicats** representent les proprietes et relations du monde. Ils peuvent etre vrais ou faux dans un etat donne.

```pddl
(:predicates
  (at ?obj - object ?loc - location)
  (in ?obj - object ?veh - vehicle)
  (connected ?from ?to - location)
  (empty ?veh - vehicle)
)
```

**Notation** :
- `?var` : Variable (commence par `?`)
- `?var - Type` : Variable avec type
- Les predicats sans parametres sont des propositions atomiques

### 2.4 Actions

Les **actions** (ou schemas d'action) definissent les transitions possibles entre etats.

```pddl
(:action load
  :parameters (?obj - object ?veh - vehicle ?loc - location)
  :precondition (and (at ?obj ?loc) (at ?veh ?loc) (empty ?veh))
  :effect (and (in ?obj ?veh) (not (at ?obj ?loc)) (not (empty ?veh)))
)
```

#### Composantes d'une action

| Composante | Description |
|------------|-------------|
| `:parameters` | Variables avec leurs types |
| `:precondition` | Conditions logiques (conjonction, disjonction, negation) |
| `:effect` | Changements : additions et suppressions de predicats |

---

## 3. Exemple complet : Domaine Logistics

Le domaine **Logistics** est un classique de la planification. Un robot doit deplacer des colis entre des lieux en utilisant des vehicules (camions, avions).

In [1]:
# Definition du domaine PDDL - Logistics
LOGISTICS_DOMAIN = """
(define (domain logistics-simple)
  (:requirements :strips :typing)
  (:types 
    location - object
    package vehicle - object
    truck - vehicle
  )
  
  (:predicates
    ;; Position des objets et vehicules
    (at-loc ?p - package ?l - location)
    (at-veh ?v - vehicle ?l - location)
    
    ;; Chargement
    (in ?p - package ?v - vehicle)
    (empty ?v - vehicle)
  )
  
  ;; Action : Charger un colis dans un vehicule
  (:action load
    :parameters (?p - package ?v - vehicle ?l - location)
    :precondition (and 
                    (at-loc ?p ?l) 
                    (at-veh ?v ?l) 
                    (empty ?v))
    :effect (and 
              (in ?p ?v) 
              (not (at-loc ?p ?l)) 
              (not (empty ?v)))
  )
  
  ;; Action : Decharger un colis d'un vehicule
  (:action unload
    :parameters (?p - package ?v - vehicle ?l - location)
    :precondition (and 
                    (in ?p ?v) 
                    (at-veh ?v ?l))
    :effect (and 
              (at-loc ?p ?l) 
              (not (in ?p ?v)) 
              (empty ?v))
  )
  
  ;; Action : Deplacer un vehicule entre deux lieux
  (:action drive
    :parameters (?v - vehicle ?from ?to - location)
    :precondition (at-veh ?v ?from)
    :effect (and 
              (at-veh ?v ?to) 
              (not (at-veh ?v ?from)))
  )
)
"""

print("Domaine PDDL defini : logistics-simple")
print("Actions disponibles : load, unload, drive")

Domaine PDDL defini : logistics-simple
Actions disponibles : load, unload, drive


### Interpretation du domaine Logistics

| Element | Description |
|---------|-------------|
| **Types** | `location`, `package`, `vehicle`, `truck` (sous-type de vehicle) |
| **Predicats** | `at-loc`, `at-veh`, `in`, `empty` |
| **load** | Charge un colis dans un vehicule (meme lieu, vehicule vide) |
| **unload** | Decharge un colis d'un vehicule |
| **drive** | Deplace un vehicule entre deux lieux |

> **Note** : Chaque action a des **preconditions** (conditions necessaires) et des **effets** (changements appliques).

---

## 4. Structure d'un fichier probleme PDDL

Le fichier **probleme** definit une instance specifique du domaine.

### 4.1 Syntaxe generale

```pddl
(define (problem nom-du-probleme)
  (:domain nom-du-domaine)
  (:objects <objets...>)
  (:init <etat-initial...>)
  (:goal <but...>)
)
```

| Section | Description |
|---------|-------------|
| `:domain` | Reference au domaine utilise |
| `:objects` | Instances concretes des types |
| `:init` | Predicats vrais a l'etat initial |
| `:goal` | Condition a atteindre |

### 4.2 Declaration des objets

```pddl
(:objects
  ;; Lieux
  depot warehouse store - location
  
  ;; Colis
  box1 box2 box3 - package
  
  ;; Vehicules
  truck1 truck2 - truck
)
```

**Syntaxe** : `obj1 obj2 obj3 - type` (plusieurs objets du meme type)

### 4.3 Etat initial

L'etat initial definit les predicats qui sont **vrais** au debut. Tous les autres sont supposes **faux** (hypothese du monde ferme).

```pddl
(:init
  ;; Positions initiales
  (at-loc box1 depot)
  (at-loc box2 depot)
  (at-veh truck1 warehouse)
  (at-veh truck2 store)
  
  ;; Vehicules vides
  (empty truck1)
  (empty truck2)
)
```

> **Hypothese du monde ferme** : Tout predicat non mentionne dans `:init` est suppose faux.

### 4.4 But

Le but definit l'etat a atteindre. C'est une formule logique (generalement une conjonction).

```pddl
(:goal (and 
  (at-loc box1 store)
  (at-loc box2 warehouse)
))
```

**Le planificateur** doit trouver une sequence d'actions qui rend tous les predicats du but vrais.

---

## 5. Exemple complet : Probleme Logistics

In [2]:
# Definition du probleme PDDL - Logistics
LOGISTICS_PROBLEM = """
(define (problem logistics-simple-1)
  (:domain logistics-simple)
  
  (:objects
    ;; Lieux
    depot warehouse store - location
    
    ;; Colis a livrer
    box1 box2 - package
    
    ;; Vehicules
    truck1 - truck
  )
  
  (:init
    ;; Positions initiales des colis
    (at-loc box1 depot)
    (at-loc box2 depot)
    
    ;; Position initiale du camion
    (at-veh truck1 depot)
    
    ;; Le camion est vide
    (empty truck1)
  )
  
  (:goal (and
    ;; box1 doit etre au store
    (at-loc box1 store)
    
    ;; box2 doit etre au warehouse
    (at-loc box2 warehouse)
  ))
)
"""

print("Probleme PDDL defini : logistics-simple-1")
print("\nEtat initial :")
print("  - box1, box2 au depot")
print("  - truck1 au depot (vide)")
print("\nBut :")
print("  - box1 au store")
print("  - box2 au warehouse")

Probleme PDDL defini : logistics-simple-1

Etat initial :
  - box1, box2 au depot
  - truck1 au depot (vide)

But :
  - box1 au store
  - box2 au warehouse


### Visualisation du probleme

| Objet | Type | Etat initial | But |
|-------|------|--------------|-----|
| `box1` | package | depot | store |
| `box2` | package | depot | warehouse |
| `truck1` | truck | depot (vide) | - |

Une solution possible :
1. `load(box1, truck1, depot)`
2. `drive(truck1, depot, store)`
3. `unload(box1, truck1, store)`
4. `drive(truck1, store, depot)`
5. `load(box2, truck1, depot)`
6. `drive(truck1, depot, warehouse)`
7. `unload(box2, truck1, warehouse)`

---

## 6. PDDL avec unified-planning

La bibliotheque **unified-planning** permet de manipuler des modeles PDDL directement en Python, sans ecrire de fichiers PDDL manuellement.

In [3]:
# Verification de unified-planning
try:
    import unified_planning as up
    from unified_planning.shortcuts import *
    print(f"unified-planning version: {up.__version__}")
    UP_AVAILABLE = True
except ImportError:
    print("ERREUR: unified-planning non installe")
    print("Installez avec: pip install unified-planning")
    UP_AVAILABLE = False

unified-planning version: 1.3.0


### 6.1 Definition des types et predicats

In [4]:
if UP_AVAILABLE:
    from collections import OrderedDict
    
    # Definition des types
    Location = UserType('Location')
    Package = UserType('Package')
    Vehicle = UserType('Vehicle')
    Truck = UserType('Truck', father=Vehicle)  # Truck herite de Vehicle
    
    # Definition des predicats (fluents)
    # Note: L'API unified-planning 1.3+ utilise OrderedDict pour la signature
    at_loc = Fluent('at_loc', BoolType(), OrderedDict([('package', Package), ('location', Location)]))
    at_veh = Fluent('at_veh', BoolType(), OrderedDict([('vehicle', Vehicle), ('location', Location)]))
    in_pkg = Fluent('in', BoolType(), OrderedDict([('package', Package), ('vehicle', Vehicle)]))
    empty = Fluent('empty', BoolType(), OrderedDict([('vehicle', Vehicle)]))
    
    print("Types definis : Location, Package, Vehicle, Truck")
    print("Predicats definis : at_loc, at_veh, in, empty")

Types definis : Location, Package, Vehicle, Truck
Predicats definis : at_loc, at_veh, in, empty


### 6.2 Definition des actions

In [5]:
if UP_AVAILABLE:
    from collections import OrderedDict
    
    # Action : load
    load = InstantaneousAction('load', OrderedDict([('package', Package), ('vehicle', Vehicle), ('location', Location)]))
    # Recuperer les parametres de l'action
    p_load = load.parameter('package')
    v_load = load.parameter('vehicle')
    l_load = load.parameter('location')
    load.add_precondition(at_loc(p_load, l_load))
    load.add_precondition(at_veh(v_load, l_load))
    load.add_precondition(empty(v_load))
    load.add_effect(at_loc(p_load, l_load), False)  # suppression
    load.add_effect(in_pkg(p_load, v_load), True)   # ajout
    load.add_effect(empty(v_load), False)           # suppression
    
    # Action : unload
    unload = InstantaneousAction('unload', OrderedDict([('package', Package), ('vehicle', Vehicle), ('location', Location)]))
    p_unload = unload.parameter('package')
    v_unload = unload.parameter('vehicle')
    l_unload = unload.parameter('location')
    unload.add_precondition(in_pkg(p_unload, v_unload))
    unload.add_precondition(at_veh(v_unload, l_unload))
    unload.add_effect(at_loc(p_unload, l_unload), True)  # ajout
    unload.add_effect(in_pkg(p_unload, v_unload), False) # suppression
    unload.add_effect(empty(v_unload), True)             # ajout
    
    # Action : drive
    drive = InstantaneousAction('drive', OrderedDict([('vehicle', Vehicle), ('from', Location), ('to', Location)]))
    v_drive = drive.parameter('vehicle')
    from_drive = drive.parameter('from')
    to_drive = drive.parameter('to')
    drive.add_precondition(at_veh(v_drive, from_drive))
    drive.add_effect(at_veh(v_drive, from_drive), False)  # suppression
    drive.add_effect(at_veh(v_drive, to_drive), True)     # ajout
    
    print("Actions definies :")
    print(f"  - load: {load.name}({', '.join(str(p) for p in load.parameters)})")
    print(f"  - unload: {unload.name}({', '.join(str(p) for p in unload.parameters)})")
    print(f"  - drive: {drive.name}({', '.join(str(p) for p in drive.parameters)})")

Actions definies :
  - load: load(Package package, Vehicle vehicle, Location location)
  - unload: unload(Package package, Vehicle vehicle, Location location)
  - drive: drive(Vehicle vehicle, Location from, Location to)


### 6.3 Creation du probleme complet

In [6]:
if UP_AVAILABLE:
    # Creation du probleme
    problem = Problem('logistics-example')
    
    # Ajout des objets
    depot = Object('depot', Location)
    warehouse = Object('warehouse', Location)
    store = Object('store', Location)
    box1 = Object('box1', Package)
    box2 = Object('box2', Package)
    truck1 = Object('truck1', Truck)
    
    problem.add_objects([depot, warehouse, store, box1, box2, truck1])
    
    # Etat initial
    problem.set_initial_value(at_loc(box1, depot), True)
    problem.set_initial_value(at_loc(box2, depot), True)
    problem.set_initial_value(at_veh(truck1, depot), True)
    problem.set_initial_value(empty(truck1), True)
    
    # But
    problem.add_goal(at_loc(box1, store))
    problem.add_goal(at_loc(box2, warehouse))
    
    # Ajout des actions au probleme
    problem.add_action(load)
    problem.add_action(unload)
    problem.add_action(drive)
    
    print("Probleme cree avec unified-planning")
    print(f"Objets: {[o.name for o in problem.all_objects]}")
    print(f"Actions: {[a.name for a in problem.actions]}")
    print(f"Buts: {[str(g) for g in problem.goals]}")

Probleme cree avec unified-planning
Objets: ['depot', 'warehouse', 'store', 'box1', 'box2', 'truck1']
Actions: ['load', 'unload', 'drive']
Buts: ['at_loc(box1, store)', 'at_loc(box2, warehouse)']


### 6.4 Resolution du probleme

In [7]:
if UP_AVAILABLE:
    from unified_planning.engines import PlanGenerationResultStatus
    
    try:
        # Utiliser un planificateur disponible
        with OneshotPlanner(name='pyperplan') as planner:
            print(f"Planificateur: {planner.name}")
            print("Resolution en cours...\n")
            
            result = planner.solve(problem)
            
            if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
                print("Solution trouvee !")
                print("=" * 50)
                for i, action in enumerate(result.plan.actions):
                    params = ', '.join(str(p.object()) for p in action.actual_parameters)
                    print(f"  {i+1}. {action.action.name}({params})")
                print("=" * 50)
                print(f"Longueur du plan: {len(result.plan.actions)} actions")
            else:
                print(f"Statut: {result.status}")
    except Exception as e:
        print(f"Erreur: {e}")
        print("\nSolution attendue (manuelle):")
        print("  1. load(box1, truck1, depot)")
        print("  2. drive(truck1, depot, store)")
        print("  3. unload(box1, truck1, store)")
        print("  4. drive(truck1, store, depot)")
        print("  5. load(box2, truck1, depot)")
        print("  6. drive(truck1, depot, warehouse)")
        print("  7. unload(box2, truck1, warehouse)")

Erreur: 

Solution attendue (manuelle):
  1. load(box1, truck1, depot)
  2. drive(truck1, depot, store)
  3. unload(box1, truck1, store)
  4. drive(truck1, store, depot)
  5. load(box2, truck1, depot)
  6. drive(truck1, depot, warehouse)
  7. unload(box2, truck1, warehouse)


### Interpretation de la solution

Le planificateur trouve une sequence d'actions qui transforme l'etat initial en etat but :

| Etape | Action | Etat apres |
|-------|--------|------------|
| 0 | (initial) | box1,box2@depot, truck1@depot |
| 1 | load(box1, truck1, depot) | box1 in truck1 |
| 2 | drive(truck1, depot, store) | truck1@store |
| 3 | unload(box1, truck1, store) | box1@store (but 1 OK) |
| 4 | drive(truck1, store, depot) | truck1@depot |
| 5 | load(box2, truck1, depot) | box2 in truck1 |
| 6 | drive(truck1, depot, warehouse) | truck1@warehouse |
| 7 | unload(box2, truck1, warehouse) | box2@warehouse (but 2 OK) |

> **Note** : Le plan a 7 actions n'est pas optimal en longueur mais satisfait le but. Un planificateur optimal pourrait trouver une solution plus courte.

---

## 7. Extensions PDDL avances

PDDL dispose de nombreuses extensions au-delà de STRIPS de base.

### 7.1 Preconditions negatives

Requirement `:negative-preconditions`

```pddl
(:action enter
  :parameters (?robot - robot ?room - location)
  :precondition (and 
                  (at ?robot ?room)
                  (not (locked ?room)))  ;; negation
  :effect (inside ?robot ?room)
)
```

### 7.2 Effets conditionnels

Requirement `:conditional-effects`

```pddl
(:action move
  :parameters (?v - vehicle ?from ?to - location)
  :precondition (at-veh ?v ?from)
  :effect (and 
            (at-veh ?v ?to) 
            (not (at-veh ?v ?from))
            ;; Effet conditionnel : si le vehicule a des passagers,
            ;; leur position change aussi
            (when (has-passengers ?v) 
                  (forall (?p - passenger) 
                    (when (in ?p ?v) 
                          (and (at ?p ?to) (not (at ?p ?from))))))
          )
)
```

### 7.3 Quantificateurs universels

Requirement `:universal-preconditions` et `:conditional-effects`

```pddl
;; Tous les colis doivent etre livres
(:goal (forall (?p - package) 
          (exists (?l - location) 
            (and (destination ?p ?l) (at-loc ?p ?l)))))
```

### 7.4 Resume des requirements

| Requirement | Description | Exemple d'usage |
|-------------|-------------|-----------------|
| `:strips` | Base STRIPS | Toujours requis |
| `:typing` | Types d'objets | `(:types truck car - vehicle)` |
| `:negative-preconditions` | `not` dans preconditions | `(not (locked ?r))` |
| `:disjunctive-preconditions` | `or` dans preconditions | `(or (at ?x A) (at ?x B))` |
| `:equality` | Test `=` | `(not (= ?x ?y))` |
| `:conditional-effects` | `when` dans effets | `(when (cond) (effect))` |
| `:adl` | ADL complet | Combine plusieurs requirements |

---

## 8. Exemple : Domaine du Gripper

Le **Gripper** est un probleme classique : un robot avec deux pinces doit deplacer des balles entre deux pieces.

In [8]:
# Domaine PDDL du Gripper (classique)
GRIPPER_DOMAIN = """
(define (domain gripper-strips)
  (:requirements :strips :typing)
  (:types room ball gripper)
  
  (:predicates
    (room ?r - room)
    (ball ?b - ball)
    (gripper ?g - gripper)
    (at-robby ?r - room)        ;; position du robot
    (at ?b - ball ?r - room)    ;; position d'une balle
    (free ?g - gripper)         ;; pince libre
    (carry ?b - ball ?g - gripper)  ;; balle tenue
  )
  
  ;; Action : deplacer le robot entre deux pieces
  (:action move
    :parameters (?from ?to - room)
    :precondition (and (room ?from) (room ?to) (at-robby ?from))
    :effect (and (at-robby ?to) (not (at-robby ?from)))
  )
  
  ;; Action : prendre une balle avec une pince
  (:action pick
    :parameters (?obj - ball ?room - room ?gripper - gripper)
    :precondition (and 
                    (ball ?obj) 
                    (room ?room) 
                    (gripper ?gripper) 
                    (at ?obj ?room) 
                    (at-robby ?room) 
                    (free ?gripper))
    :effect (and 
              (carry ?obj ?gripper) 
              (not (at ?obj ?room)) 
              (not (free ?gripper)))
  )
  
  ;; Action : poser une balle
  (:action drop
    :parameters (?obj - ball ?room - room ?gripper - gripper)
    :precondition (and 
                    (ball ?obj) 
                    (room ?room) 
                    (gripper ?gripper) 
                    (at-robby ?room) 
                    (carry ?obj ?gripper))
    :effect (and 
              (at ?obj ?room) 
              (free ?gripper) 
              (not (carry ?obj ?gripper)))
  )
)
"""

print("Domaine Gripper defini")
print("Actions: move, pick, drop")

Domaine Gripper defini
Actions: move, pick, drop


Definition du probleme PDDL correspondant au domaine Gripper, instanciant les objets concrets, l'etat initial et le but a atteindre.

In [9]:
# Probleme PDDL du Gripper
GRIPPER_PROBLEM = """
(define (problem strips-gripper2)
  (:domain gripper-strips)
  (:objects 
    rooma roomb - room
    ball1 ball2 - ball
    left right - gripper
  )
  (:init 
    ;; Definition des objets (optionnel dans certains planificateurs)
    (room rooma) (room roomb)
    (ball ball1) (ball ball2)
    (gripper left) (gripper right)
    
    ;; Position initiale du robot
    (at-robby rooma)
    
    ;; Pinces libres
    (free left) (free right)
    
    ;; Positions initiales des balles
    (at ball1 rooma)
    (at ball2 rooma)
  )
  (:goal (and 
    (at ball1 roomb)
    (at ball2 roomb)
  ))
)
"""

print("Probleme Gripper defini")
print("\nEtat initial:")
print("  - Robot dans rooma")
print("  - ball1, ball2 dans rooma")
print("  - Pinces left, right libres")
print("\nBut:")
print("  - ball1 et ball2 dans roomb")

Probleme Gripper defini

Etat initial:
  - Robot dans rooma
  - ball1, ball2 dans rooma
  - Pinces left, right libres

But:
  - ball1 et ball2 dans roomb


### Analyse du probleme Gripper

| Element | Valeur |
|---------|--------|
| **Objets** | 2 rooms, 2 balls, 2 grippers |
| **Etat initial** | Robot + balles dans rooma |
| **But** | Balles dans roomb |
| **Actions possibles** | move, pick (x2), drop (x2) |

Une solution optimale :
1. `pick(ball1, rooma, left)`
2. `pick(ball2, rooma, right)`
3. `move(rooma, roomb)`
4. `drop(ball1, roomb, left)`
5. `drop(ball2, roomb, right)`

> **Observation** : Le robot peut porter deux balles simultanement car il a deux pinces !

---

## 9. Exercices

---

## 10. Resume

### Points cles

| Concept | Description |
|---------|-------------|
| **PDDL** | Langage standard pour la planification automatique |
| **Domaine** | Definit types, predicats et actions (reutilisable) |
| **Probleme** | Definit objets, etat initial et but (specifique) |
| **Predicats** | Proprietes/relations binaires (vrai/faux) |
| **Actions** | Transitions avec preconditions et effets |
| **STRIPS** | Modele de base (add/delete lists) |

### Structure PDDL

```
domain.pddl          problem.pddl
    |                     |
    v                     v
(:types ...)        (:objects ...)
(:predicates ...)   (:init ...)
(:action ...)       (:goal ...)
```

### Prochaines etapes

Dans le notebook suivant **Planners-3-State-Space**, nous explorerons :
- La representation des espaces d'etats
- La recherche dans un graphe d'etats
- Les algorithmes de recherche (BFS, DFS, A*)

---

## Ressources

- [PDDL Reference](https://planning.wiki/) - Documentation complete
- [unified-planning](https://github.com/aiplan4eu/unified-planning) - Bibliotheque Python
- [IPC Benchmarks](https://github.com/aibasel/downward-benchmarks) - Problemes PDDL standards
- [Fast Downward](https://www.fast-downward.org/) - Planificateur de reference

---

**Notebook suivant** : [Planners-3-State-Space](Planners-3-State-Space.ipynb)